# Aria Frontend Development Guide

**Complete guide for building web interfaces, UI components, styling, and user interactions.**

Learn how to build responsive, accessible web interfaces for Aria.

## Frontend Architecture

### Technology Stack

```
Frontend Layer (Browser)
├─ React 18+ (component framework)
├─ TypeScript (type safety)
├─ Tailwind CSS (utility styling)
├─ Redux/Context API (state management)
├─ Vite (build tool)
└─ Socket.io (real-time communication)

UI Components
├─ Chat Interface (messages, input)
├─ Aria Character Display (3D/CSS animations)
├─ Dashboard (metrics, graphs)
├─ Settings Panel
└─ Status Indicators

APIs Consumed
├─ /api/chat (streaming SSE)
├─ /api/tts (text-to-speech)
├─ /api/aria/command (character commands)
├─ /api/ai/status (health check)
└─ /api/quantum/* (quantum jobs)
```

### Project Structure

```
apps/
├─ aria/                          # Main Aria character interface
│  ├─ index.html                  # Static HTML
│  ├─ aria_controller.js          # Character control logic
│  ├─ auto-execute.html           # Command interface
│  ├─ styles.css                  # Aria styling
│  └─ server.py                   # Python backend (port 8080)
│
├─ chat/                          # Chat web interface
│  ├─ src/
│  │  ├─ components/
│  │  │  ├─ ChatWindow.tsx        # Main chat UI
│  │  │  ├─ MessageList.tsx       # Messages display
│  │  │  ├─ InputBox.tsx          # User input
│  │  │  └─ StatusIndicator.tsx   # Connection status
│  │  ├─ hooks/
│  │  │  ├─ useChat.ts            # Chat logic hook
│  │  │  ├─ useSSE.ts             # Server-sent events
│  │  │  └─ useTTS.ts             # Text-to-speech
│  │  ├─ utils/
│  │  │  ├─ api.ts                # API client
│  │  │  ├─ storage.ts            # Local storage
│  │  │  └─ formatting.ts         # Text formatting
│  │  ├─ App.tsx                  # Root component
│  │  └─ main.tsx                 # Entry point
│  ├─ public/                     # Static assets
│  ├─ package.json
│  └─ vite.config.ts
│
└─ dashboard/                     # Admin dashboard
   ├─ src/
   │  ├─ components/
   │  │  ├─ MetricsPanel.tsx
   │  │  ├─ ChartView.tsx
   │  │  └─ SystemHealth.tsx
   │  └─ App.tsx
   └─ package.json
```

---

## Component Development

### Creating a React Component

**Button Component Example:**

```typescript
// src/components/Button.tsx
import React from 'react';
import './Button.css';

interface ButtonProps {
  label: string;
  onClick: () => void;
  variant?: 'primary' | 'secondary' | 'danger';
  disabled?: boolean;
  loading?: boolean;
}

export const Button: React.FC<ButtonProps> = ({
  label,
  onClick,
  variant = 'primary',
  disabled = false,
  loading = false,
}) => {
  return (
    <button
      className={`button button--${variant}`}
      onClick={onClick}
      disabled={disabled || loading}
    >
      {loading ? '...' : label}
    </button>
  );
};
```

**Styling with Tailwind CSS:**

```typescript
// Using Tailwind directly in components
export const ChatMessage: React.FC<ChatMessageProps> = ({ message, sender }) => {
  return (
    <div className={`flex ${sender === 'user' ? 'justify-end' : 'justify-start'}`}>
      <div
        className={`max-w-xs px-4 py-2 rounded-lg ${
          sender === 'user'
            ? 'bg-blue-500 text-white'
            : 'bg-gray-300 text-black'
        }`}
      >
        {message}
      </div>
    </div>
  );
};
```

---

## State Management

### Using React Hooks

**Simple State with useState:**

```typescript
import { useState } from 'react';

export const Counter = () => {
  const [count, setCount] = useState(0);
  
  return (
    <div>
      <p>Count: {count}</p>
      <button onClick={() => setCount(count + 1)}>Increment</button>
    </div>
  );
};
```

**Custom Hook for Chat:**

```typescript
// src/hooks/useChat.ts
import { useState, useCallback } from 'react';

interface Message {
  id: string;
  text: string;
  sender: 'user' | 'assistant';
  timestamp: number;
}

export const useChat = () => {
  const [messages, setMessages] = useState<Message[]>([]);
  const [loading, setLoading] = useState(false);
  const [error, setError] = useState<string | null>(null);

  const sendMessage = useCallback(async (text: string) => {
    setLoading(true);
    setError(null);

    try {
      const response = await fetch('/api/chat', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ message: text }),
      });

      if (!response.ok) throw new Error('Failed to send message');

      const data = await response.json();
      setMessages((prev) => [
        ...prev,
        { id: Date.now().toString(), text, sender: 'user', timestamp: Date.now() },
        { id: (Date.now() + 1).toString(), text: data.response, sender: 'assistant', timestamp: Date.now() },
      ]);
    } catch (err) {
      setError((err as Error).message);
    } finally {
      setLoading(false);
    }
  }, []);

  return { messages, loading, error, sendMessage };
};
```

**Global State with Context API:**

```typescript
// src/context/ChatContext.tsx
import React, { createContext, useContext, useState } from 'react';

interface ChatContextType {
  theme: 'light' | 'dark';
  setTheme: (theme: 'light' | 'dark') => void;
}

const ChatContext = createContext<ChatContextType | undefined>(undefined);

export const ChatProvider: React.FC<{ children: React.ReactNode }> = ({ children }) => {
  const [theme, setTheme] = useState<'light' | 'dark'>('light');

  return (
    <ChatContext.Provider value={{ theme, setTheme }}>
      {children}
    </ChatContext.Provider>
  );
};

export const useChat = () => {
  const context = useContext(ChatContext);
  if (!context) throw new Error('useChat must be used within ChatProvider');
  return context;
};
```

---

## API Integration

### Server-Sent Events (SSE) for Streaming

**Hook for SSE streaming:**

```typescript
// src/hooks/useSSE.ts
import { useEffect, useState, useCallback } from 'react';

export const useSSE = (url: string) => {
  const [data, setData] = useState<string[]>([]);
  const [error, setError] = useState<string | null>(null);
  const [loading, setLoading] = useState(false);

  const startStream = useCallback((body: object) => {
    setLoading(true);
    setData([]);
    setError(null);

    const eventSource = new EventSource(
      url + '?' + new URLSearchParams({ body: JSON.stringify(body) })
    );

    eventSource.onmessage = (event) => {
      if (event.data === '[DONE]') {
        eventSource.close();
        setLoading(false);
      } else {
        try {
          const chunk = JSON.parse(event.data);
          setData((prev) => [...prev, chunk.content || chunk]);
        } catch {
          setData((prev) => [...prev, event.data]);
        }
      }
    };

    eventSource.onerror = () => {
      setError('Stream error');
      eventSource.close();
      setLoading(false);
    };
  }, [url]);

  return { data, error, loading, startStream };
};

// Usage
const { data, startStream } = useSSE('/api/chat/stream');
const handleChat = () => {
  startStream({ message: userInput });
};
```

**Fetch with streaming:**

```typescript
export const fetchWithStream = async (
  url: string,
  onChunk: (chunk: string) => void
) => {
  const response = await fetch(url);
  const reader = response.body?.getReader();

  if (!reader) throw new Error('No response body');

  while (true) {
    const { done, value } = await reader.read();
    if (done) break;
    onChunk(new TextDecoder().decode(value));
  }
};
```

---

## Styling Best Practices

### Tailwind CSS Patterns

**Responsive Design:**

```html
<!-- Mobile first, then desktop -->
<div class="grid grid-cols-1 md:grid-cols-2 lg:grid-cols-3 gap-4">
  <div class="p-4 bg-blue-500">Item 1</div>
  <div class="p-4 bg-blue-500">Item 2</div>
  <div class="p-4 bg-blue-500">Item 3</div>
</div>

<!-- Tailwind breakpoints:
  sm: 640px
  md: 768px
  lg: 1024px
  xl: 1280px
  2xl: 1536px
-->
```

**Dark Mode:**

```typescript
export const DarkModeComponent = () => {
  return (
    <div className="bg-white dark:bg-gray-900 text-black dark:text-white">
      <p>This adapts to dark mode</p>
    </div>
  );
};
```

**Custom CSS for complex animations:**

```css
/* src/styles/animations.css */
@keyframes slideIn {
  from {
    transform: translateX(-100%);
    opacity: 0;
  }
  to {
    transform: translateX(0);
    opacity: 1;
  }
}

.message-enter {
  animation: slideIn 0.3s ease-out;
}
```

---

## Performance Optimization

### Code Splitting & Lazy Loading

```typescript
import { lazy, Suspense } from 'react';

const ChatPage = lazy(() => import('./pages/ChatPage'));
const Dashboard = lazy(() => import('./pages/Dashboard'));

export const App = () => {
  return (
    <Suspense fallback={<LoadingSpinner />}>
      <Routes>
        <Route path="/chat" element={<ChatPage />} />
        <Route path="/dashboard" element={<Dashboard />} />
      </Routes>
    </Suspense>
  );
};
```

### Memoization

```typescript
import { memo, useMemo, useCallback } from 'react';

// Prevent unnecessary re-renders
const MessageItem = memo(({ message }: { message: Message }) => {
  return <div>{message.text}</div>;
});

export const MessageList = ({ messages }: { messages: Message[] }) => {
  // Memoize computed values
  const sortedMessages = useMemo(
    () => [...messages].sort((a, b) => a.timestamp - b.timestamp),
    [messages]
  );

  // Memoize callbacks
  const handleClick = useCallback((id: string) => {
    console.log('Clicked:', id);
  }, []);

  return (
    <div>
      {sortedMessages.map((msg) => (
        <MessageItem key={msg.id} message={msg} />
      ))}
    </div>
  );
};
```

---

## Testing Frontend Code

### Component Testing with Vitest

```typescript
// src/components/__tests__/Button.test.tsx
import { describe, it, expect, vi } from 'vitest';
import { render, screen } from '@testing-library/react';
import { Button } from '../Button';

describe('Button', () => {
  it('renders button with label', () => {
    render(<Button label="Click me" onClick={() => {}} />);
    expect(screen.getByText('Click me')).toBeInTheDocument();
  });

  it('calls onClick when clicked', () => {
    const onClick = vi.fn();
    render(<Button label="Click" onClick={onClick} />);
    screen.getByText('Click').click();
    expect(onClick).toHaveBeenCalled();
  });

  it('disables button when disabled prop is true', () => {
    render(<Button label="Click" onClick={() => {}} disabled={true} />);
    expect(screen.getByText('Click')).toBeDisabled();
  });
});
```


## Frontend Development Workflow

### Setup Local Development

```bash
# Install Node.js 18+
node --version  # v18+

# Install dependencies
npm install

# Start dev server with HMR (Hot Module Replacement)
npm run dev
# Open http://localhost:5173

# Build for production
npm run build

# Preview production build
npm run preview
```

### Development Commands

```bash
# Run tests
npm test

# Run tests with coverage
npm test -- --coverage

# Lint code
npm run lint

# Format code
npm run format

# Type check
npm run type-check

# Build production bundle
npm run build

# Analyze bundle size
npm run build -- --analyze
```

---

## Accessibility (a11y)

### WCAG 2.1 Compliance

**Semantic HTML:**

```html
<!-- Good: Semantic HTML -->
<header>
  <nav>
    <ul>
      <li><a href="/">Home</a></li>
      <li><a href="/about">About</a></li>
    </ul>
  </nav>
</header>

<!-- Bad: Non-semantic -->
<div id="header">
  <div id="nav">
    <div id="menu">
      <div><a href="/">Home</a></div>
    </div>
  </div>
</div>
```

**ARIA Labels:**

```typescript
// Use ARIA for interactive elements
<button
  aria-label="Close dialog"
  aria-pressed={isOpen}
  onClick={toggleDialog}
>
  ✕
</button>

// Form labels
<label htmlFor="email">Email:</label>
<input id="email" type="email" required />
```

**Keyboard Navigation:**

```typescript
const SearchInput = () => {
  const [results, setResults] = useState([]);
  const [selectedIndex, setSelectedIndex] = useState(-1);

  const handleKeyDown = (e: React.KeyboardEvent) => {
    switch (e.key) {
      case 'ArrowDown':
        setSelectedIndex(Math.min(selectedIndex + 1, results.length - 1));
        break;
      case 'ArrowUp':
        setSelectedIndex(Math.max(selectedIndex - 1, -1));
        break;
      case 'Enter':
        // Select result
        break;
    }
  };

  return (
    <input
      type="search"
      onKeyDown={handleKeyDown}
      aria-autocomplete="list"
      aria-expanded={results.length > 0}
    />
  );
};
```

---

## Frontend Best Practices Checklist

### Code Quality
- [ ] TypeScript strict mode enabled
- [ ] No `any` types used
- [ ] ESLint all passing
- [ ] Prettier formatting applied
- [ ] 80%+ test coverage

### Performance
- [ ] Bundle size < 500KB (gzipped)
- [ ] Lazy load routes
- [ ] Memoize expensive computations
- [ ] Images optimized & lazy loaded
- [ ] No N+1 API calls

### Accessibility
- [ ] WCAG 2.1 AA compliant
- [ ] Keyboard navigation works
- [ ] Color contrast >4.5:1
- [ ] All interactive elements labeled
- [ ] Works with screen readers

### Security
- [ ] No hardcoded secrets
- [ ] Input sanitized
- [ ] CSRF tokens on forms
- [ ] Content Security Policy set
- [ ] Secure headers configured

### User Experience
- [ ] Loading states shown
- [ ] Error messages helpful
- [ ] Responsive design (mobile-first)
- [ ] Fast initial load (<2s)
- [ ] Graceful degradation
